# 質問応答のデータセットを読み込んで中身を確認する (JSQuAD)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_qa/qa_dataset.ipynb)

NLP クックブック 20-01 ／ 質問応答

質問応答の学習や評価に使う日本語のデータセットを読み込み、質問・文書・答えがどんな形で入っているかを確かめたい。

In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("sbintuitions/JSQuAD")     # JGLUE の読解データセット。Wikipedia の段落と、それに対する質問・答え
print(ds)
ex = ds["validation"][0]
for k, v in ex.items():
    print(f"{k}: {v}")

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'is_impossible'],
        num_rows: 62859
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'is_impossible'],
        num_rows: 4442
    })
})
id: a10336p0q0
title: 梅雨
context: 梅雨 [SEP] 梅雨（つゆ、ばいう）は、北海道と小笠原諸島を除く日本、朝鮮半島南部、中国の南部から長江流域にかけての沿海部、および台湾など、東アジアの広範囲においてみられる特有の気象現象で、5月から7月にかけて来る曇りや雨の多い期間のこと。雨季の一種である。
question: 日本で梅雨がないのは北海道とどこか。
answers: {'answer_start': [25, 25, 25], 'text': ['小笠原諸島', '小笠原諸島を除く日本', '小笠原諸島']}
is_impossible: False


In [3]:
# 答えは「文書中の文字位置」で与えられる。位置と文字列が合っているかを確かめる
ok = 0
for ex in ds["validation"].select(range(500)):
    for text, start in zip(ex["answers"]["text"], ex["answers"]["answer_start"]):
        ok += ex["context"][start:start + len(text)] == text
print("位置が合っている答え:", ok, "/", sum(len(ex["answers"]["text"]) for ex in ds["validation"].select(range(500))))

位置が合っている答え: 1451 / 1451


In [4]:
import pandas as pd

df = ds["validation"].to_pandas()
df["context_len"] = df["context"].str.len()
df["question_len"] = df["question"].str.len()
df["answer_len"] = df["answers"].map(lambda a: len(a["text"][0]))
df["n_answers"] = df["answers"].map(lambda a: len(set(a["text"])))
print(df[["context_len", "question_len", "answer_len", "n_answers"]].describe().round(1).loc[["mean", "50%", "max"]])

# 質問の種類を、疑問詞でざっくり数える
import re
kinds = {"誰": r"誰|だれ", "いつ": r"いつ|何年|何月|何日|何世紀", "どこ": r"どこ|何処|何県|何市|どの国", "いくつ": r"いくつ|何人|何個|何回|いくら|どのくらい|何メートル", "何": r"何|なに|なん"}
def kind(q):
    return next((k for k, pat in kinds.items() if re.search(pat, q)), "その他")
print(df["question"].map(kind).value_counts())
print(df.sample(3, random_state=0)[["question"]].assign(answer=df["answers"].map(lambda a: a["text"][0])).to_string(index=False))

      context_len  question_len  answer_len  n_answers
mean        183.5          28.8         6.2        1.3
50%         169.0          25.0         5.0        1.0
max         906.0          98.0        19.0        3.0
question
その他    1703
何      1515
いつ      583
どこ      307
誰       214
いくつ     120
Name: count, dtype: int64
                                        question    answer
何年6月、アメリカに亡命したミャオ族の元王国軍将軍とアメリカ軍退役少佐によるクーデター計画が発覚      2007
                           中国の北朝について書かれた歴史書は何ですか 『北史』（ほくし）
                         各種行政事務の非能率の原因ともなっていたのは？      住居表示


- SQuAD 形式: `context`（文書）、`question`（質問）、`answers`（答えの文字列と開始位置のリスト）。評価データには 1 つの質問に複数の正解が付いていることがある。
- JSQuAD の `context` は「記事タイトル [SEP] 段落」の形になっている。タイトルを使わないなら `[SEP]` より後ろを取り出す。
- JSQuAD の質問はすべて答えが文書中にある（SQuAD 1.1 相当）。答えのない質問を含む設定はこの章の「文書に答えがない質問を「答えなし」と判定する」のレシピを参照。
- 最近の `datasets` は、読み込みスクリプトを使う古い形式のデータセットを読めない。Parquet 形式で公開されているリポジトリを選ぶ（ここでは `sbintuitions/JSQuAD`）。